## Cell 1


In [ ]:
!pip -q install python-docx

## Cell 2


In [ ]:
from google.colab import files

uploaded = files.upload()

md_filename = next(iter(uploaded))

print("Uploaded:", md_filename)
print("Ready to convert!")

## Cell 3


In [ ]:
import re
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# Create document
doc = Document()

# Page margins
section = doc.sections[0]
section.top_margin = Inches(0.7)
section.bottom_margin = Inches(0.7)
section.left_margin = Inches(0.8)
section.right_margin = Inches(0.8)

# Document title
doc.add_heading("Colab Notebook Export", 0)

# Code formatting function
def add_code_block(lines):
    paragraph = doc.add_paragraph()
    paragraph.paragraph_format.left_indent = Inches(0.15)
    paragraph.paragraph_format.space_after = Pt(8)

    shading = OxmlElement("w:shd")
    shading.set(qn("w:fill"), "F0F0F0")
    paragraph._p.get_or_add_pPr().append(shading)

    run = paragraph.add_run("\n".join(lines))
    run.font.name = "Consolas"
    run.font.size = Pt(9)
    run.font.color.rgb = RGBColor(40, 40, 40)

# Read Markdown
with open(md_filename, "r", encoding="utf-8") as f:
    content = f.read()

lines = content.splitlines()

in_code = False
code_lines = []

for line in lines:

    # Start of code block
    if line.startswith("```"):
        if not in_code:
            in_code = True
            code_lines = []
        else:
            add_code_block(code_lines)
            in_code = False
        continue

    if in_code:
        code_lines.append(line)
        continue

    # Original notebook code-cell numbering
    match = re.match(r"^### Code cell (\d+)", line)

    if match:
        doc.add_heading(
            f"Code Cell {match.group(1)}",
            level=2
        )
        continue

    # Markdown headings
    heading = re.match(r"^(#{1,6})\s+(.+)", line)

    if heading:
        level = min(len(heading.group(1)), 3)
        doc.add_heading(heading.group(2), level=level)
        continue

    # Bullet points
    if line.startswith("- ") or line.startswith("* "):
        doc.add_paragraph(
            line[2:],
            style="List Bullet"
        )
        continue

    # Numbered lists
    if re.match(r"^\d+\.\s+", line):
        text = re.sub(r"^\d+\.\s+", "", line)
        doc.add_paragraph(text, style="List Number")
        continue

    # Normal text
    if line.strip():
        doc.add_paragraph(line)

# Save document
output_path = "/content/Colab_Notebook_Export.docx"

doc.save(output_path)

print("DOCUMENT CREATED SUCCESSFULLY!")
print("Location:", output_path)

# Download
files.download(output_path)